In [1]:
import shogi
from shogi import CSA

In [2]:
from bs4 import BeautifulSoup

In [3]:
import requests

In [4]:
import io

In [5]:
from datetime import date, timedelta, datetime

In [6]:
import time

In [7]:
import re

In [8]:
import pickle

### Download competition page (https://golan.sakura.ne.jp/denryusen/dr1_test4a/dr1_live.php)

In [9]:
# def get_denryu_links(live_url):
#     page = requests.get(live_url)
#     soup = BeautifulSoup(page.content, 'html.parser')
#     links = soup.find_all('a')
#     urls = []
#     for link in links:
#         href = link.attrs['href']
#         if 'denryusen_single' in href and 'suisho' in href:
#             match_id = href.split('#')[-1]

#             # url
#             csa_address = "https://golan.sakura.ne.jp/denryusen/dr1_test4a/kifufiles/" + match_id + '.csa'
#             urls.append(csa_address)
#         elif 'dr1test1' in href and 'suisho' in href:
#             match_id = href.split('/')[-1].split('.')[-2]

#             # url
#             csa_address = "https://golan.sakura.ne.jp/denryusen/dr1_test4a/kifufiles/" + match_id + '.csa'
#             urls.append(csa_address)
#     print(len(urls))
#     return urls

# denryu_urls = get_denryu_links("https://golan.sakura.ne.jp/denryusen/dr1_test1/dr1_live.php")
# denryu_urls += get_denryu_links("https://golan.sakura.ne.jp/denryusen/dr1_test2/dr1_live.php")
# denryu_urls += get_denryu_links("https://golan.sakura.ne.jp/denryusen/dr1_test3/dr1_live.php")
# denryu_urls = get_denryu_links("https://golan.sakura.ne.jp/denryusen/dr1_test4a/dr1_live.php")
# len(denryu_urls)

### Iterate over matches

#### Download CSA record if interested

#### Convert CSA to SFEN

#### Extract meta data from the record

#### Merge SFEN and meta data

#### Accumulate SFEN

In [10]:
def get_eval(csa):
    values = []
    for m in re.finditer("'\*\*\s+(-?\d+)", csa, re.MULTILINE):
        values.append(int(m.group(1)))
        
#     return [values[0::2], values[1::2]]
    return values

In [11]:
def get_sfen(url):
    # Download
    csa_page = requests.get(url)
    csa = csa_page.content.decode()

    # Convert CSA to SFEN
    try:
        sfen = CSA.Parser.parse_str(csa)[0]
        sfen['csa'] = csa
        sfen['url'] = url
        sfen['record'] = 'sfen ' + sfen['sfen'] + ' moves ' + ' '.join(sfen['moves'])
        sfen['values'] = get_eval(csa)
        sfen['toryo'] = (re.search("%TORYO|%KACHI", csa) != None)
#         print(sfen['values'])
#         print(len(sfen['moves']), len(sfen['values']), sfen['toryo'])
    
        return sfen
    except:
        print(csa)
        return None


In [12]:
# with open('dr_black.sfen', 'w') as fb, open('dr_white.sfen', 'w') as fw:
#     for csa_url in denryu_urls:
#         sfen = get_sfen(csa_url)

#         # Write out
#         print(sfen['names'])
#         if 'suisho' == sfen['names'][0] and sfen['win'] == 'b':
#             fb.write(sfen['record'] + '\n')
#         if 'suisho' == sfen['names'][1] and sfen['win'] == 'w':
#             fw.write(sfen['record'] + '\n')


### Download Floodgate records

#### Download ratings

In [13]:
rating_page = requests.get('http://wdoor.c.u-tokyo.ac.jp/shogi/logs/LATEST/players-floodgate.html')

In [14]:
soup = BeautifulSoup(rating_page.content, 'html.parser')

In [15]:
names = soup.select('td.name a')
ratings =  soup.select('td.rate span')
print(len(names), len(ratings))
assert(len(names) == len(ratings))

engine_names = []

for i in range(len(names)):
    name = names[i].contents[0]
    try:
        rating = int(ratings[i].contents[0])
    except:
        rating = -9999
    if rating >= 3600:
        engine_names.append(name)
        print(name, rating)


444 444
Suisho201003_TR3990X 4630
LUSI 4518
QNSkai_TR3990X 4497
Suisho200914_TR3990X 4477
Suisho2kai_TR3990X 4447
BURNING_BRIDGES 4414
WHITE_BELG 4399
FROZEN_BRIDGE 4396
Nao. 4394
Marulk 4388
CRAZY-DOCTOR 4376
Rade_Berger 4355
Hinatsuru_Ai 4340
CrazyQueen 4326
BlackCat_021b 4304
test1d 4291
QueenAI_test201010_i9-7920x_12c 4285
Beluga 4281
ECLIPSE_RyzenTR3990X 4279
LUNA 4270
Kamuy201005_RyzenTR3990X 4269
ANESIS 4266
Black_Cat 4253
Incinerator 4252
BLUETRANSPARENCY 4252
Fomalhaut 4235
tanupon_TRX40A 4232
Mariel 4221
beer2020 4218
daigo8 4217
first_test_20200901 4213
Yashajin_Ai 4213
Kamuy_z005_i9-7980XE_18c 4208
suikyuwomen 4203
BlackCat_027av017_i9-7980XE_18c 4188
mbk 4187
ECLIPSE_24t 4187
BlackCat_027av017_i7-6700HQ 4183
Sagittarius 4170
ECLIPSE_16t 4169
Lladro 4153
Abe 4151
TEST 4143
Nashi 4136
ideal_light 4131
BlackCat_027av077_i7-6700HQ 4130
ideal_white 4114
AXIF 4113
ry4500U 4109
19W 4105
Reaper 4091
Unagi 4091
BlackCat_008kai9_i9-7980X_18c 4091
dlshogitest 4090
EliNe_23 4090
10W 4

#### Download records

In [16]:
fg_urls = []
for i in range(0, 365):
    time.sleep(1)
    d = date.today() - timedelta(days=i)
    
    # Download daily index page
    daily_url = ('http://wdoor.c.u-tokyo.ac.jp/shogi/x/' + 
                 str(d.year) +'/' + str(d.month).zfill(2) + '/' + str(d.day).zfill(2) + '/');
#     print(daily_url)
    daily_page = requests.get(daily_url)
    soup = BeautifulSoup(daily_page.content, 'html.parser')
#     print(daily_page)
    
    # Iterate over links
    links = soup.find_all('a')
    for link in links:
        href = link.attrs['href']
#         print(href)
        if '.csa' in href:
            _ = href.split('+')
            black_name = _[2]
            white_name = _[3]
            if (black_name in engine_names) and (white_name in engine_names):   
                fg_urls.append(daily_url + href)
            
    print(str(d.year) +'/' + str(d.month).zfill(2) + '/' + str(d.day).zfill(2) + '/', len(fg_urls))
    
print(len(fg_urls))

2020/11/16/ 11
2020/11/15/ 50
2020/11/14/ 75
2020/11/13/ 104
2020/11/12/ 169
2020/11/11/ 235
2020/11/10/ 302
2020/11/09/ 355
2020/11/08/ 435
2020/11/07/ 528
2020/11/06/ 597
2020/11/05/ 664
2020/11/04/ 728
2020/11/03/ 873
2020/11/02/ 1017
2020/11/01/ 1135
2020/10/31/ 1228
2020/10/30/ 1358
2020/10/29/ 1479
2020/10/28/ 1540
2020/10/27/ 1664
2020/10/26/ 1771
2020/10/25/ 1872
2020/10/24/ 1984
2020/10/23/ 2101
2020/10/22/ 2247
2020/10/21/ 2396
2020/10/20/ 2504
2020/10/19/ 2609
2020/10/18/ 2679
2020/10/17/ 2756
2020/10/16/ 2823
2020/10/15/ 2868
2020/10/14/ 2969
2020/10/13/ 3072
2020/10/12/ 3155
2020/10/11/ 3244
2020/10/10/ 3303
2020/10/09/ 3353
2020/10/08/ 3362
2020/10/07/ 3432
2020/10/06/ 3492
2020/10/05/ 3566
2020/10/04/ 3657
2020/10/03/ 3702
2020/10/02/ 3756
2020/10/01/ 3825
2020/09/30/ 3898
2020/09/29/ 3977
2020/09/28/ 4048
2020/09/27/ 4114
2020/09/26/ 4169
2020/09/25/ 4257
2020/09/24/ 4332
2020/09/23/ 4399
2020/09/22/ 4474
2020/09/21/ 4544
2020/09/20/ 4620
2020/09/19/ 4683
2020/09/18/ 47

In [1]:
def make_teacher(sfen, minply=16):
    board = shogi.Board()
    t = ""
    for i in range(len(sfen['values'])):
        if (i+1) >= minply:
            t += 'sfen ' + board.sfen() + '\n';
            t += 'move ' + sfen['moves'][i] + '\n';
            t += 'ply ' + str(i+1) + '\n';
            sfen['values'][i] = max(-32767, sfen['values'][i])
            sfen['values'][i] = min(32767, sfen['values'][i])
            if (i % 2) == 0:
                # black's turn
                t += 'result ' + ('1' if sfen['win'] == 'b' else '-1') + '\n'
                t += 'score ' + str(sfen['values'][i]) + '\n';
            else:
                t += 'result ' + ('1' if sfen['win'] == 'w' else '-1') + '\n'
                t += 'score ' + str(-sfen['values'][i]) + '\n';
            t += 'e' + '\n';
        board.push(shogi.Move.from_usi(sfen['moves'][i]))
    return t

In [18]:
game_count = 0
def print_sfen(sfen):
    print(sfen['names'])
    print('position ' + sfen['record'] + '\n')
    
with open('teacher.txt', 'w') as ft:
    teachers = []
    
#     SFENs = []
#     for csa_url in fg_urls:
#         sfen = get_sfen(csa_url)
#         SFENs.append(sfen)

    with open('sfens.pickle', 'rb') as fp:
        SFENs = pickle.load(fp)
        
    for sfen in SFENs:

        if (sfen == None) or (not sfen['toryo']):
            # Could not parse CSA file or no contest;
            continue
            
        if game_count >= 10000:
            break
            
        teacher = make_teacher(sfen, minply=20)
        #print(make_teacher(sfen))
        ft.write(make_teacher(sfen))
        
        if game_count % 100 == 0:
            print(datetime.now(), game_count)
        game_count += 1
        
#         time.sleep(0.2)
        
print('game_count:', game_count)

# with open('sfens.pickle', 'wb') as f:
#     # Pickle the 'SFENs' using the highest protocol available.
#     pickle.dump(SFENs, f, pickle.HIGHEST_PROTOCOL)

2020-11-16 09:41:01.136666 0
2020-11-16 09:41:03.988029 100
2020-11-16 09:41:07.114417 200
2020-11-16 09:41:10.106851 300
2020-11-16 09:41:13.283892 400
2020-11-16 09:41:16.596717 500
2020-11-16 09:41:20.195062 600
2020-11-16 09:41:23.530707 700
2020-11-16 09:41:26.916144 800
2020-11-16 09:41:30.026475 900
2020-11-16 09:41:33.013740 1000
2020-11-16 09:41:36.072560 1100
2020-11-16 09:41:39.453520 1200
2020-11-16 09:41:42.345626 1300
2020-11-16 09:41:45.488297 1400
2020-11-16 09:41:48.292869 1500
2020-11-16 09:41:51.028360 1600
2020-11-16 09:41:54.080782 1700
2020-11-16 09:41:57.546652 1800
2020-11-16 09:42:01.332557 1900
2020-11-16 09:42:04.703526 2000
2020-11-16 09:42:08.231446 2100
2020-11-16 09:42:11.817497 2200
2020-11-16 09:42:15.609260 2300
2020-11-16 09:42:19.220973 2400
2020-11-16 09:42:22.865289 2500
2020-11-16 09:42:26.455186 2600
2020-11-16 09:42:29.825624 2700
2020-11-16 09:42:33.600285 2800
2020-11-16 09:42:37.270743 2900
2020-11-16 09:42:40.838516 3000
2020-11-16 09:42:44.

#### Make binary teacher file

In [19]:
# learn convert_bin output_file_name [出力ファイル名] [入力ファイル名1] [入力ファイル名2]
! (cd /home/hmatsuya/workspace/Shogi/YaneuraOuOriginal/exe && ./YaneuraOu-by-gcc learn convert_bin output_file_name /home/hmatsuya/workspace/Shogi/cobra2019b/notebook/teacher.bin /home/hmatsuya/workspace/Shogi/cobra2019b/notebook/teacher.txt , quit)

learn command , learn from /home/hmatsuya/workspace/Shogi/cobra2019b/notebook/teacher.txt , 
base dir        : 
target dir      : 
info string Hash table allocation: Linux Large Pages used.
info string eHash Clear begin , Hash size =  128[MB]
info string eHash Clear done.
info string EvalDirectory = ./eval
info string loading eval file : eval/nn.bin
info string read book file : book/standard_book.db
info string read book done.
info string Hash Clear begin , Hash size =  16[MB]
info string Hash Clear done.
convert_bin..
convert /home/hmatsuya/workspace/Shogi/cobra2019b/notebook/teacher.txt ... done
all done
/bin/bash: line 1: 31185 Floating point exception(core dumped) ./YaneuraOu-by-gcc learn convert_bin output_file_name /home/hmatsuya/workspace/Shogi/cobra2019b/notebook/teacher.bin /home/hmatsuya/workspace/Shogi/cobra2019b/notebook/teacher.txt , quit


#### Learn from the binary teacher file

In [20]:
! (cd /home/hmatsuya/workspace/Shogi/YaneuraOuOriginal/exe && ./YaneuraOu-by-gcc learn /home/hmatsuya/workspace/Shogi/cobra2019b/notebook/teacher.bin , quit)

learn command , learn from /home/hmatsuya/workspace/Shogi/cobra2019b/notebook/teacher.bin , 
base dir        : 
target dir      : 
loop              : 1
eval_limit        : 32000
save_only_once    : false
no_shuffle        : false
Loss Function     : ELMO_METHOD(WCSC27)
mini-batch size   : 1000000
nn_batch_size     : 1000
nn_options        : 
learning rate     : 0 , 0 , 0
eta_epoch         : 0 , 0
scheduling        : default
discount rate     : 0
reduction_gameply : 1
LAMBDA            : 0.33
LAMBDA2           : 0.33
LAMBDA_LIMIT      : 32000
mirror_percentage : 0
eval_save_interval  : 1000000000 sfens
loss_output_interval: 1000000 sfens
init..
info string Hash table allocation: Linux Large Pages used.
info string eHash Clear begin , Hash size =  128[MB]
info string eHash Clear done.
info string EvalDirectory = ./eval
info string loading eval file : eval/nn.bin
info string read book file : book/standard_book.db
info string read book done.
info string Hash Clear begin , Hash size =  16[

### Make book from SFEN files (white & black)

In [21]:
!pwd

/home/hmatsuya/workspace/Shogi/cobra2019b/notebook


In [22]:
# ! (cd ../exe && ./YaneuraOu-by-gcc bench , quit)

In [23]:
# ! (cd ../exe && ./YaneuraOu-by-gcc makebook from_sfen bw ../notebook/dr_black.sfen ../notebook/dr_white.sfen ../notebook/denryu.db moves 512 , quit)

In [24]:
# ! sed 's/position //g' fl_black.sfen > fg_black.sfen

In [25]:
# ! sed 's/position //g' fl_white.sfen > fg_white.sfen

In [26]:
# ! (cd ../exe && ./YaneuraOu-by-gcc makebook from_sfen bw ../notebook/fg_black.sfen ../notebook/fg_white.sfen ../notebook/floodgate.db moves 512 , quit)

### Merge book files